# Experiment 1.3.9 — Phase-aware Cross-user Contrastive Local Feature Learning (revised)

## Why this revision exists

The first 1.3.9 run showed a clear optimization pathology: applying supervised contrastive loss from epoch 1 caused L3 firing rate to jump from the normal `~0.08–0.10` regime to `~0.33–0.38`, while both train and validation classification performance collapsed. The failure therefore cannot yet be interpreted as evidence that cross-user contrastive learning is unsuitable for this task.

This revision keeps the same scientific hypothesis but changes the optimization protocol so that silent / near-silent local spike-count vectors do not dominate contrastive training.

The final deployment target is unchanged:

\[X_{spike} \rightarrow SNN \rightarrow z_1,z_2,\ldots,z_B, \qquad z_b\in\mathbb{R}^{64}.\]

No Raw branch and no contrastive head are retained at inference.

## Revised contrastive protocol

The SNN architecture is still `30 → 128 → 128 → 64`, shifts `((2,3),(2,3),(2))`, `tau_mem=22.54 ms`, threshold `0.5`, continuous state through the gesture, and no reset at 250 ms boundaries.

Three conditions remain:

- **A — `cls_only`**: whole-gesture CE only.
- **B — `con250`**: CE + phase-aware cross-user SupCon on active 250 ms local features.
- **C — `con500`**: CE + the same loss on an active 500 ms motif formed by concatenating adjacent 250 ms features. The deployed representation is still the original 250 ms 64-D feature.

Positive pairs remain **same label + same relative-phase bucket + different user**. Negatives remain **different label + same phase bucket**. Only fully-valid windows participate.

## Optimization safeguards

The revised run makes four changes.

1. **CE-only warm-up**: epochs `1–25` use only gesture classification loss, allowing the SNN to enter a normal firing regime before any contrastive gradient is applied.
2. **Gradual contrastive ramp**: during the next 10 epochs the effective contrastive weight rises linearly from 0 to the selected `lambda_con`; after that it stays fixed.
3. **Active-item gating**: a 250/500 ms item is contrasted only if its L3 spike-count representation contains at least one spike. Silent motifs are excluded before normalization.
4. **Consistent transform**: both 250 ms and 500 ms representations use `log1p(...)` followed by L2 normalization.

The training objective after warm-up is

\[L = L_{cls} + \lambda_{eff}(e)L_{SupCon}.\]

where `lambda_eff(e)=0` during warm-up and then ramps to the target value.

## Hyperparameter selection and leakage control

Temperature remains fixed at `0.1`. The development grid is now

`lambda_con ∈ {0, 0.01, 0.03, 0.10}`.

Including `0` is essential: if no contrastive weight beats the CE-only control in mean validation SNN250 fresh-probe balanced accuracy across seeds `(11,23,101)`, the experiment explicitly selects `lambda=0` rather than choosing the least-bad nonzero value. The test split is not touched during this selection stage.

This revised run writes to a separate artifact directory `active_warmup_v2`, so the original failed 1.3.9 checkpoints/results are preserved for comparison and cannot be accidentally resumed.

In [ ]:
%run -i ../scripts/experiment_1_3_9_phase_aware_contrastive/01_setup.py

## Model and training

The implementation now records both **active-item fraction** and **valid-anchor fraction among active items**, in addition to L1/L2/L3 firing rates and dead-neuron fractions. This distinguishes metadata coverage from actual non-silent SNN representation coverage.

In [ ]:
%run -i ../scripts/experiment_1_3_9_phase_aware_contrastive/02_model_training.py

## Development sweep and final confirmation

Primary representation quality is still measured with a frozen SNN plus a fresh train-only-standardized Logistic Regression probe. Logistic-regression `C` is selected on validation from `{1e-3, 1e-2, 1e-1, 1, 10}`.

Primary readout: **SNN250**. Secondary readout: **SNN125**. Final test evaluation occurs only after validation-only selection of `lambda_con`.

In [ ]:
%run -i ../scripts/experiment_1_3_9_phase_aware_contrastive/03_run_experiment.py

## Diagnostics

The diagnostic stage now matches the revised training definition: retrieval uses active motifs only. It reports cross-user phase-aware retrieval/kNN, Raw250+SNN fusion (diagnostic only), validation BA versus epoch, L3 firing-rate trajectories, active-item coverage, and valid-anchor coverage.

The central sanity check is whether contrastive regularization can improve representation quality **without pushing L3 firing rate far outside the CE-only regime**.

In [ ]:
%run -i ../scripts/experiment_1_3_9_phase_aware_contrastive/04_diagnostics.py

## Decision rules

- If a nonzero `lambda_con` beats `lambda=0` in mean validation SNN250 probe BA and retains a reasonable firing regime, phase-aware contrastive shaping is supported.
- If `con500` consistently outperforms `con250`, 250 ms features likely behave as local primitives that become class-relevant only after short temporal composition.
- If `lambda=0` wins after the warm-up/gating fix, the negative result is much stronger: class-conditioned local SupCon itself is probably mismatched to this task, rather than merely suffering from silent-vector normalization.
- Raw+SNN fusion remains diagnostic only and is never treated as the final architecture.